# 四模型全面对比: SenseVoice & Fun-ASR-Nano (基线 vs 微调)

在**全量**验证集上串行评估四个模型：
1. **SenseVoice-base**: 未微调基线
2. **SenseVoice-ft**: 微调后
3. **Nano-base**: 未微调基线
4. **Nano-ft**: 微调后

适配 Mac MPS 本地环境，逐个模型加载/评估/释放，避免内存不足。

## 1. 环境检查

In [1]:
import warnings
import logging
logging.getLogger('root').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

import os, re, json, time, gc
import torch

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')

import funasr
print(f'FunASR: {funasr.__version__}')

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    print(f'matplotlib: {matplotlib.__version__}')
except ImportError:
    print('matplotlib 未安装，可视化步骤将跳过')

import editdistance
print(f'editdistance: OK')

print('\n--- 路径检查 ---')
paths = {
    'SenseVoice 微调': os.path.expanduser('~/Projects/Agent/model/sensevoice_lora/model.pt.best'),
    'Nano 微调': os.path.expanduser('~/Projects/Agent/model/funasr_nano_v3/model.pt.best'),
    '验证集': os.path.expanduser('~/Desktop/hengdong_asr_trainset/manifests/val.jsonl'),
    '长音频数据': os.path.expanduser('~/Desktop/hengdong_asr_trainset/manifests/extra_long_talks.jsonl'),
}
for name, path in paths.items():
    print(f'  {name}: {"OK" if os.path.exists(path) else "NOT FOUND"} - {path}')

PyTorch: 2.11.0
Device: mps
FunASR: 1.3.1
matplotlib: 3.10.9
editdistance: OK

--- 路径检查 ---
  SenseVoice 微调: OK - /Users/fanhua/Projects/Agent/model/sensevoice_lora/model.pt.best
  Nano 微调: OK - /Users/fanhua/Projects/Agent/model/funasr_nano_v3/model.pt.best
  验证集: OK - /Users/fanhua/Desktop/hengdong_asr_trainset/manifests/val.jsonl
  长音频数据: OK - /Users/fanhua/Desktop/hengdong_asr_trainset/manifests/extra_long_talks.jsonl


## 2. 工具函数与数据加载

In [2]:
def clean_sensevoice_text(text):
    """去除 SenseVoice 特殊标记"""
    return re.sub(r'<\|[^|]*\|>', '', text).strip()

def levenshtein(s1, s2):
    """计算编辑距离（循环版本，避免递归栈溢出）"""
    if len(s1) < len(s2):
        s1, s2 = s2, s1
    if len(s2) == 0:
        return len(s1)
    prev = list(range(len(s2) + 1))
    for c1 in s1:
        curr = [prev[0] + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j + 1] + 1, curr[j] + 1, prev[j] + (c1 != c2)))
        prev = curr
    return prev[-1]

def free_memory():
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

print('工具函数就绪')

工具函数就绪


In [3]:
# 加载全量验证集
DATA_ROOT = os.path.expanduser('~/Desktop/hengdong_asr_trainset')
VAL_JSONL = os.path.join(DATA_ROOT, 'manifests/val.jsonl')

samples = []
with open(VAL_JSONL, 'r', encoding='utf-8') as f:
    for line in f:
        s = json.loads(line)
        audio = s.get('audio_filepath') or s.get('source')
        if audio and os.path.exists(audio):
            samples.append(s)

print(f'验证集: {len(samples)} 条有效样本')
print(f'\n前3条:')
for s in samples[:3]:
    audio = s.get('audio_filepath') or s.get('source')
    text = s.get('text') or s.get('target')
    print(f'  {os.path.basename(audio)} -> {text}')

验证集: 2261 条有效样本

前3条:
  backend_user_1769774772513_1_1.wav -> 爱
  backend_user_1769774772513_20_20.wav -> 东西
  backend_user_1769774772513_24_24.wav -> 多


## 3. 串行评估函数

In [4]:
from funasr import AutoModel

SV_CKPT = os.path.expanduser('~/Projects/Agent/model/sensevoice_lora/model.pt.best')
NANO_CKPT = os.path.expanduser('~/Projects/Agent/model/funasr_nano_v3/model.pt.best')

def eval_sensevoice(model, samples, label):
    """评估 SenseVoice 模型"""
    results = []
    total_cer, total_chars, exact = 0.0, 0, 0
    start = time.time()
    for i, s in enumerate(samples):
        audio = s.get('audio_filepath') or s.get('source')
        expected = s.get('text') or s.get('target')
        if not audio or not os.path.exists(audio):
            continue
        try:
            res = model.generate(input=audio, language='auto', use_itn=True)
            pred = clean_sensevoice_text(res[0]['text']) if res else ''
        except:
            pred = ''
        dist = levenshtein(expected, pred)
        ref_len = max(len(expected), 1)
        cer = dist / ref_len
        total_cer += dist
        total_chars += ref_len
        if expected == pred:
            exact += 1
        results.append({'id': i, 'expected': expected, 'predicted': pred, 'cer': cer})
        if (i + 1) % 100 == 0:
            print(f'    {label}: {i+1}/{len(samples)} | CER={total_cer/total_chars:.2%} | exact={exact}')
    cer = total_cer / total_chars if total_chars > 0 else 0
    elapsed = time.time() - start
    return {'name': label, 'results': results, 'cer': cer, 'exact': exact, 'total': len(results), 'time': elapsed}

def eval_nano(model, samples, label):
    """评估 Fun-ASR-Nano 模型"""
    results = []
    total_cer, total_chars, exact = 0.0, 0, 0
    start = time.time()
    with torch.inference_mode():
        for i, s in enumerate(samples):
            audio = s.get('audio_filepath') or s.get('source')
            expected = s.get('text') or s.get('target')
            if not audio or not os.path.exists(audio):
                continue
            try:
                res = model.generate(input=audio, language='中文', itn=True)
                pred = res[0]['text'] if res else ''
            except:
                pred = ''
            dist = levenshtein(expected, pred)
            ref_len = max(len(expected), 1)
            cer = dist / ref_len
            total_cer += dist
            total_chars += ref_len
            if expected == pred:
                exact += 1
            results.append({'id': i, 'expected': expected, 'predicted': pred, 'cer': cer})
            if (i + 1) % 100 == 0:
                print(f'    {label}: {i+1}/{len(samples)} | CER={total_cer/total_chars:.2%} | exact={exact}')
    cer = total_cer / total_chars if total_chars > 0 else 0
    elapsed = time.time() - start
    return {'name': label, 'results': results, 'cer': cer, 'exact': exact, 'total': len(results), 'time': elapsed}

print('评估函数就绪')

评估函数就绪


## 4. 评估 [1/4] SenseVoice-base

In [5]:
# all_results = {}

# print('[1/4] SenseVoice-base')
# model = AutoModel(model='iic/SenseVoiceSmall', disable_update=True, device=device)
# sv_base_res = eval_sensevoice(model, samples, 'SV-base')
# all_results['SV-base'] = sv_base_res
# print(f'\n  完成! CER={sv_base_res["cer"]:.2%} | exact={sv_base_res["exact"]}/{sv_base_res["total"]} | 耗时={sv_base_res["time"]:.1f}s')

# del model
# free_memory()

## 5. 评估 [2/4] SenseVoice-ft (微调)

In [6]:
# print('[2/4] SenseVoice-ft (微调)')
# model = AutoModel(model='iic/SenseVoiceSmall', lora_only=True, disable_update=True, device=device)
# ckpt = torch.load(SV_CKPT, map_location='cpu', weights_only=False)
# keys_matched = len(set(model.model.state_dict().keys()) & set(ckpt['state_dict'].keys()))
# model.model.load_state_dict(ckpt['state_dict'], strict=False)
# del ckpt
# print(f'  微调权重已载入 (keys: {keys_matched})')

# sv_ft_res = eval_sensevoice(model, samples, 'SV-ft')
# all_results['SV-ft'] = sv_ft_res
# print(f'\n  完成! CER={sv_ft_res["cer"]:.2%} | exact={sv_ft_res["exact"]}/{sv_ft_res["total"]} | 耗时={sv_ft_res["time"]:.1f}s')

# del model
# free_memory()

## 6. 评估 [3/4] Nano-base

In [7]:
# print('[3/4] Nano-base')
# model = AutoModel(model='FunAudioLLM/Fun-ASR-Nano-2512', hub='ms', device=device, disable_update=True)

# nano_base_res = eval_nano(model, samples, 'Nano-base')
# all_results['Nano-base'] = nano_base_res
# print(f'\n  完成! CER={nano_base_res["cer"]:.2%} | exact={nano_base_res["exact"]}/{nano_base_res["total"]} | 耗时={nano_base_res["time"]:.1f}s')

# del model
# free_memory()

## 7. 评估 [4/4] Nano-ft (微调)

In [8]:
# print('[4/4] Nano-ft (微调)')
# model = AutoModel(
#     model='FunAudioLLM/Fun-ASR-Nano-2512',
#     hub='ms', device=device, disable_update=True,
#     init_param=NANO_CKPT,
# )
# print('  微调权重已载入')

# nano_ft_res = eval_nano(model, samples, 'Nano-ft')
# all_results['Nano-ft'] = nano_ft_res
# print(f'\n  完成! CER={nano_ft_res["cer"]:.2%} | exact={nano_ft_res["exact"]}/{nano_ft_res["total"]} | 耗时={nano_ft_res["time"]:.1f}s')

# del model
# free_memory()

## 12. 长音频测试 (SenseVoice-ft + VAD)

使用 VAD + SenseVoice-ft pipeline 测试长录音，评估真实连续语音表现。

In [9]:
import editdistance

PUNCT_PATTERN = re.compile(
    r'[\s\.,!?;:\-()\[\]{}\u3000\uff0c\u3002\uff01\uff1f\u3001\uff1b\uff1a'
    r'\u201c\u201d\u2018\u2019\uff08\uff09\u3010\u3011\u300a\u300b]'
)

def normalize_text(text):
    return PUNCT_PATTERN.sub('', text)

def compute_cer(hyp, ref):
    hyp, ref = normalize_text(hyp), normalize_text(ref)
    if not ref:
        return 0.0
    return editdistance.eval(hyp, ref) / len(ref)

print('加载 SenseVoice-ft...')
long_model = AutoModel(
    model='iic/SenseVoiceSmall', lora_only=True, disable_update=True,
    device=device,
)
long_ckpt = torch.load(SV_CKPT, map_location='cpu', weights_only=False)
long_model.model.load_state_dict(long_ckpt['state_dict'], strict=False)
del long_ckpt

# 单独加载 VAD 模型用于分段
print('加载 VAD 模型...')
vad_model = AutoModel(
    model='iic/speech_fsmn_vad_zh-cn-16k-common-pytorch',
    device=device,
    disable_update=True,
)
print('VAD + ASR 模型加载成功!')

加载 SenseVoice-ft...
funasr version: 1.3.1.
加载 VAD 模型...
funasr version: 1.3.1.
VAD + ASR 模型加载成功!


In [10]:
# 加载长音频数据
LONG_JSONL = os.path.join(DATA_ROOT, 'manifests/extra_long_talks.jsonl')
long_samples = []
with open(LONG_JSONL) as f:
    for line in f:
        item = json.loads(line)
        audio = item['audio_filepath']
        if audio.startswith('/mnt/data/'):
            audio = audio.replace('/mnt/data/hengdong_asr_trainset', DATA_ROOT)
        long_samples.append({
            'id': item['id'],
            'audio': audio,
            'text': item['text'],
            'duration': item['duration'],
            'speaker': item.get('speaker_id', 'unknown'),
        })

print(f'长音频: {len(long_samples)} 条')
for s in long_samples:
    exists = os.path.exists(s['audio'])
    print(f'  {s["speaker"]} | {s["duration"]/60:.1f}min | {"OK" if exists else "NOT FOUND"}')

长音频: 6 条
  talks_方言老女 | 27.1min | OK
  talks_方言老男 | 30.8min | OK
  talks_方言青女 | 23.3min | OK
  talks_方言青男 | 23.3min | OK
  dialogs_方言多人 | 16.7min | OK
  dialogs_方言多人 | 9.8min | OK


In [11]:
REPORT_DIR = os.path.expanduser('~/Projects/Agent/local')

def detect_speech_segments(audio_path, vad_model, max_single_segment_time=30000, min_segment_length=100):
    """使用 VAD 检测语音段
    
    Args:
        audio_path: 音频文件路径
        vad_model: VAD AutoModel
        max_single_segment_time: 最大段时长(毫秒)，超过则截断
        min_segment_length: 最小段时长(毫秒)，过滤噪声
    
    Returns:
        list of dict: [{'start': ms, 'end': ms, 'duration': ms}, ...]
    """
    import soundfile as sf
    
    # 读取音频
    speech, sample_rate = sf.read(audio_path)
    if speech.ndim > 1:
        speech = speech[:, 0]  # 单声道
    
    # VAD 检测
    vad_res = vad_model.generate(input=speech)
    
    # 解析 VAD 结果
    # 格式: {'key': '...', 'value': [[start_ms, end_ms], [start_ms, end_ms], ...]}
    if isinstance(vad_res, list) and len(vad_res) > 0 and isinstance(vad_res[0], dict):
        time_ranges = vad_res[0].get('value', [])
    else:
        time_ranges = []
    
    print(f"    VAD 检测到 {len(time_ranges)} 个时间片段")
    
    segments = []
    for seg in time_ranges:
        start, end = int(seg[0]), int(seg[1])  # 已经是毫秒
        duration = end - start
        
        # 过滤过短的段
        if duration < min_segment_length:
            continue
        
        # 如果超过最大长度，拆分
        if duration > max_single_segment_time:
            numplits = (duration + max_single_segment_time - 1) // max_single_segment_time
            for i in range(numplits):
                s = start + i * max_single_segment_time
                e = min(start + (i + 1) * max_single_segment_time, end)
                segments.append({'start': s, 'end': e, 'duration': e - s})
        else:
            segments.append({'start': start, 'end': end, 'duration': duration})
    
    return segments

# 运行长音频测试，手动分段 + 逐段识别
long_results = []
print('开始长音频测试...')
start_long = time.time()

for i, s in enumerate(long_samples):
    if not os.path.exists(s['audio']):
        print(f'  [{i+1}] 跳过: 文件不存在')
        continue
    print(f'  [{i+1}/{len(long_samples)}] {s["speaker"]} ({s["duration"]/60:.1f}min)')
    
    # 1. VAD 分段
    print(f'    VAD 分段中...')
    vad_segments = detect_speech_segments(s['audio'], vad_model)
    print(f'    有效语音段: {len(vad_segments)} 个')
    
    # 2. 逐段识别
    segments = []
    all_pred_text = []
    
    if len(vad_segments) == 0:
        print(f'    警告: VAD 未检测到语音段，跳过')
        long_results.append({
            'id': s['id'],
            'speaker': s['speaker'],
            'audio': s['audio'],
            'duration_min': s['duration'] / 60,
            'cer': 1.0,
            'ref_len': len(normalize_text(s['text'])),
            'pred_len': 0,
            'expected': s['text'],
            'predicted': '',
            'segments': [],
        })
        continue
    
    # 预加载音频数据，避免重复读取
    import soundfile as sf
    full_speech, sr = sf.read(s['audio'])
    if full_speech.ndim > 1:
        full_speech = full_speech[:, 0]
    
    for seg_idx, seg in enumerate(vad_segments):
        # 提取片段 (VAD返回的是毫秒，需要转为采样点)
        start_sample = int(seg['start'] / 1000.0 * sr)
        end_sample = int(seg['end'] / 1000.0 * sr)
        segment_speech = full_speech[start_sample:end_sample]
        
        # 跳过太短的片段
        if len(segment_speech) < 1600:  # 小于 100ms (16kHz)
            continue
        
        # 识别
        try:
            res = long_model.generate(input=segment_speech, language='auto', use_itn=True)
            seg_text = clean_sensevoice_text(res[0]['text']) if res else ''
        except Exception as e:
            seg_text = ''
        
        segments.append({
            'idx': seg_idx,
            'text': seg_text,
            'start': seg['start'],
            'end': seg['end'],
            'duration': seg['duration'],
            'cer': 0.0,
        })
        all_pred_text.append(seg_text)
        
        if (seg_idx + 1) % 50 == 0:
            print(f'    已识别 {seg_idx+1}/{len(vad_segments)} 段')
    
    pred_text = ''.join(all_pred_text)
    cer = compute_cer(pred_text, s['text'])
    
    long_results.append({
        'id': s['id'],
        'speaker': s['speaker'],
        'audio': s['audio'],
        'duration_min': s['duration'] / 60,
        'cer': cer,
        'ref_len': len(normalize_text(s['text'])),
        'pred_len': len(normalize_text(pred_text)),
        'expected': s['text'],
        'predicted': pred_text,
        'segments': segments,
    })
    print(f'    CER: {cer:.1%} | 标注: {long_results[-1]["ref_len"]}字 | 识别: {long_results[-1]["pred_len"]}字')

long_elapsed = time.time() - start_long

# 汇总
sep = '=' * 60
print(f'\n{sep}')
print(f'长音频测试汇总 (耗时 {long_elapsed:.1f}s)')
print(f'{sep}')
if long_results:
    avg_cer = sum(r['cer'] for r in long_results) / len(long_results)
    print(f'{"说话人":<20} {"时长":>6} {"CER":>8} {"标注字数":>8} {"识别字数":>8}')
    print('-' * 55)
    for r in long_results:
        print(f'{r["speaker"]:<20} {r["duration_min"]:>5.1f}m {r["cer"]:>7.1%} {r["ref_len"]:>8} {r["pred_len"]:>8}')
    print('-' * 55)
    print(f'{"平均":<20} {"":>6} {avg_cer:>7.1%}')

    # 保存
    long_eval_path = os.path.join(REPORT_DIR, 'long_audio_eval.json')
    with open(long_eval_path, 'w', encoding='utf-8') as f:
        json.dump({
            'model': 'SenseVoice-ft (LoRA)', 'total': len(long_results),
            'avg_cer': round(avg_cer, 4), 'time': round(long_elapsed, 1),
            'results': long_results,
        }, f, ensure_ascii=False, indent=2)
    print(f'\n结果已保存: {long_eval_path}')

del long_model
free_memory()

开始长音频测试...
  [1/6] talks_方言老女 (27.1min)
    VAD 分段中...


rtf_avg: 0.240: 100%|██████████| 1/1 [00:02<00:00,  2.00s/it]                                                                                          


    VAD 检测到 373 个时间片段
    有效语音段: 373 个


rtf_avg: 0.015: 100%|██████████| 1/1 [00:00<00:00, 21.76it/s]                                                                                          


    已识别 50/373 段


...

    CER: 446.3% | 标注: 311字 | 识别: 1480字
  [6/6] dialogs_方言多人 (9.8min)
    VAD 分段中...


rtf_avg: 0.015: 100%|██████████| 1/1 [00:00<00:00,  1.49it/s]                                                                                          


    VAD 检测到 30 个时间片段
    有效语音段: 39 个


rtf_avg: 0.007: 100%|██████████| 1/1 [00:00<00:00,  5.30it/s]                                                                                          


    CER: 677.9% | 标注: 204字 | 识别: 1494字

长音频测试汇总 (耗时 120.7s)
说话人                      时长      CER     标注字数     识别字数
-------------------------------------------------------
talks_方言老女            27.1m  392.4%      909     3978
talks_方言老男            30.8m  219.9%     1268     3622
talks_方言青女            23.3m  860.0%      500     4471
talks_方言青男            23.3m  272.3%     1515     4493
dialogs_方言多人          16.7m  446.3%      311     1480
dialogs_方言多人           9.8m  677.9%      204     1494
-------------------------------------------------------
平均                           478.1%

结果已保存: /Users/fanhua/Projects/Agent/local/long_audio_eval.json


## 13. 长音频可视化报告

生成交互式 HTML 报告，展示每条长音频的识别效果。


In [12]:
# 生成长音频 HTML 报告（含分段音频）
from difflib import SequenceMatcher
import html as html_mod

def long_char_diff(ref, hyp):
    sm = SequenceMatcher(None, ref, hyp)
    h_parts = []
    for op, i1, i2, j1, j2 in sm.get_opcodes():
        t = html_mod.escape(hyp[j1:j2])
        if op == 'equal':
            h_parts.append(t)
        elif op == 'replace':
            h_parts.append(f'<span class="sub">{t}</span>')
        elif op == 'insert':
            h_parts.append(f'<span class="ins">{t}</span>')
    return ''.join(h_parts)

def cer_badge_long(c):
    if c < 0.1: return f'<span class="b ok">{c:.0%}</span>'
    if c < 0.5: return f'<span class="b warn">{c:.0%}</span>'
    return f'<span class="b bad">{c:.0%}</span>'

AUDIO_ROOT = os.path.expanduser("~/Desktop/hengdong_asr_trainset/audio_extra")

def audio_to_file_url(abs_path):
    """转换为 file:// URL 供浏览器直接访问"""
    if os.path.exists(abs_path):
        return f"file://{abs_path}"
    return ""

def format_timestamp(ms):
    """毫秒转为 MM:SS.ms 格式"""
    s = ms / 1000
    m = int(s // 60)
    sec = s % 60
    return f'{m:02d}:{sec:05.2f}'

def build_segment_audio_tag(audio_path, start_ms, end_ms):
    """生成带时间范围的 audio 标签"""
    abs_url = audio_to_file_url(audio_path)
    if not abs_url:
        return "无音频"
    start_sec = start_ms / 1000.0
    end_sec = end_ms / 1000.0
    # 使用 HTML5 audio 的 #t=[start,end] 格式指定播放范围
    src_with_range = f'{abs_url}#t={start_sec},{end_sec}'
    return f'<audio controls preload="metadata"><source src="{src_with_range}"></audio>'

# 构建报告
rows = []
seg_rows = []  # 分段详情

for r in long_results:
    exp_n = normalize_text(r["expected"])
    pred_n = normalize_text(r["predicted"])
    abs_url = audio_to_file_url(r["audio"])
    audio_tag = f'<audio controls preload="none"><source src="{abs_url}"></audio>' if abs_url else "无音频"
    diff_html = long_char_diff(exp_n, pred_n)
    
    # 主行
    rows.append(
        f'<tr class="main-row" onclick="toggleSegments(\'{r["id"]}\')">'
        f'<td class="id">{r["speaker"]}</td>'
        f'<td>{r["duration_min"]:.1f}m</td>'
        f'<td>{cer_badge_long(r["cer"])}</td>'
        f'<td class="ref">{html_mod.escape(exp_n[:80])}...</td>'
        f'<td><span class="t">{diff_html}</span></td>'
        f'<td>{audio_tag}</td>'
        f'<td class="seg-toggle">▶ {len(r["segments"])}段</td>'
        f'</tr>'
    )
    
    # 分段详情行
    if r["segments"]:
        seg_rows.append(f'<tr class="seg-header" id="hdr-{r["id"]}"><td colspan="7">📂 {r["speaker"]} - 分段详情 ({len(r["segments"])}段) <span onclick="toggleSegments(\'{r["id"]}\')" style="cursor:pointer;color:#1890ff">[收起]</span></td></tr>')
        for seg in r["segments"]:
            # 只在有有效时间戳时显示分段音频
            if seg['start'] > 0 or seg['end'] > seg['start']:
                seg_audio = build_segment_audio_tag(r["audio"], seg['start'], seg['end'])
            else:
                seg_audio = '<span class="no-seg">无时间戳</span>'
            seg_rows.append(
                f'<tr class="seg-row" id="seg-{r["id"]}-{seg["idx"]}">'
                f'<td class="seg-idx">[{seg["idx"]}]</td>'
                f'<td>{format_timestamp(seg["start"])} - {format_timestamp(seg["end"])}</td>'
                f'<td>{seg["duration"]/1000:.1f}s</td>'
                f'<td class="seg-text" colspan="2">{html_mod.escape(seg["text"] or "(无文本)")}</td>'
                f'<td>{seg_audio}</td>'
                f'<td></td>'
                f'</tr>'
            )

avg_cer = sum(r["cer"] for r in long_results) / len(long_results) if long_results else 0
summary_cards = (
    f'<div class="card"><h3>SenseVoice-ft (长音频)</h3>'
    f'<p class="big">{avg_cer:.1%}</p>'
    f'<p>CER | {len(long_results)} 条</p></div>'
)

long_html = f"""<!DOCTYPE html>
<html lang="zh-CN">
<head><meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>长音频识别报告 - SenseVoice-ft</title>
<style>
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:-apple-system,"PingFang SC","Microsoft YaHei",sans-serif;background:#f0f2f5;padding:20px}}
h1{{text-align:center;margin-bottom:20px}}
.summary{{display:flex;gap:12px;justify-content:center;margin-bottom:20px}}
.card{{background:#fff;padding:16px 28px;border-radius:8px;box-shadow:0 1px 3px rgba(0,0,0,.1);text-align:center}}
.card h3{{font-size:12px;color:#888;margin-bottom:6px}}
.card .big{{font-size:28px;font-weight:bold;color:#1890ff}}
table{{width:100%;border-collapse:collapse;background:#fff;box-shadow:0 1px 3px rgba(0,0,0,.1)}}
th{{background:#1a1a2e;color:#fff;padding:10px 12px;text-align:left;font-weight:500}}
td{{padding:10px 12px;border-bottom:1px solid #f0f0f0;vertical-align:top}}
.id{{font-weight:600;color:#333}}
.ref{{font-weight:500;color:#555;max-width:200px;word-break:break-all;font-size:13px}}
.t{{font-family:monospace;font-size:13px;line-height:1.6;display:inline-block}}
.b{{display:inline-block;padding:2px 8px;border-radius:3px;font-size:12px;font-weight:700}}
.b.ok{{background:#f6ffed;color:#52c41a;border:1px solid #b7eb8f}}
.b.warn{{background:#fffbe6;color:#faad14;border:1px solid #ffe58f}}
.b.bad{{background:#fff2f0;color:#ff4d4f;border:1px solid #ffccc7}}
.sub{{background:#ffe0b2;padding:0 1px;border-radius:2px;color:#e65100}}
.ins{{background:#c8e6c9;padding:0 1px;border-radius:2px;color:#1b5e20}}
audio{{height:28px;width:160px}}
.main-row{{cursor:pointer;background:#fafafa}}
.main-row:hover{{background:#f0f0f0}}
.seg-toggle{{color:#1890ff;font-weight:600}}
.seg-header{{background:#e6f7ff}}
.seg-row{{background:#fff}}
.seg-idx{{color:#888;font-size:12px;width:50px}}
.seg-text{{font-family:monospace;font-size:12px;word-break:break-all}}
.no-seg{{color:#999;font-size:12px}}
</style>
<script>
function toggleSegments(id) {{
    var rows = document.querySelectorAll('[id^="seg-' + id + '-"]');
    var hdr = document.getElementById('hdr-' + id);
    rows.forEach(function(row) {{
        row.style.display = row.style.display === 'none' ? 'table-row' : 'none';
    }});
    if (hdr) {{
        hdr.style.display = hdr.style.display === 'none' ? 'table-row' : 'none';
    }}
}}
</script>
</head><body>
<h1>长音频识别报告 (SenseVoice-ft + VAD)</h1>
<div class="summary">{summary_cards}</div>
<table><thead><tr><th>说话人</th><th>时长</th><th>CER</th><th>参考文本</th><th>识别结果对比</th><th>音频</th><th>分段</th></tr></thead>
<tbody>
{"".join(rows)}
{"".join(seg_rows)}
</tbody></table>
</body></html>"""

long_report_path = os.path.join(REPORT_DIR, "long_audio_report.html")
with open(long_report_path, "w", encoding="utf-8") as f:
    f.write(long_html)
print(f"长音频 HTML 报告已生成: {long_report_path}")
print(f"点击主表行可展开/收起分段详情")

长音频 HTML 报告已生成: /Users/fanhua/Projects/Agent/local/long_audio_report.html
点击主表行可展开/收起分段详情
